In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

import lightgbm as lgb
import joblib

In [ ]:
#定位项目路径
CURRENT_DIR = Path.cwd()

if (CURRENT_DIR / "data").exists() and (
    (CURRENT_DIR / "notebook").exists() or (CURRENT_DIR / "notebooks").exists()
):
    PROJECT_DIR = CURRENT_DIR
elif (CURRENT_DIR.parent / "data").exists() and (
    (CURRENT_DIR.parent / "notebook").exists() or (CURRENT_DIR.parent / "notebooks").exists()
):
    PROJECT_DIR = CURRENT_DIR.parent
else:
    raise FileNotFoundError(
        "Cannot locate project root. Please check your current working directory."
    )

DATA_RAW_DIR = PROJECT_DIR / "data" / "raw"

REPORT_DIR = PROJECT_DIR / "reports" / "model" / "lgbm"
TABLE_DIR = REPORT_DIR / "tables"
FIGURE_DIR = REPORT_DIR / "figures"
SUBMISSION_DIR = PROJECT_DIR / "submissions"
MODEL_DIR = PROJECT_DIR / "models"

for path in [REPORT_DIR, TABLE_DIR, FIGURE_DIR, SUBMISSION_DIR, MODEL_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Data directory:", DATA_RAW_DIR)
print("Report directory:", REPORT_DIR)

In [ ]:
#读取数据
train_path = DATA_RAW_DIR / "train.csv"
test_path = DATA_RAW_DIR / "test.csv"
sample_submission_path = DATA_RAW_DIR / "sample_submission.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
sample_submission = pd.read_csv(sample_submission_path)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Sample submission shape:", sample_submission.shape)

display(train_df.head())
display(test_df.head())
display(sample_submission.head())

In [ ]:
#构造颜色指数特征
def add_color_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add astronomical color index features.
    """
    df = df.copy()

    required_cols = ["u", "g", "r", "i", "z"]
    missing_cols = [col for col in required_cols if col not in df.columns]

    if missing_cols:
        raise ValueError(f"Missing columns for color features: {missing_cols}")

    df["u_g"] = df["u"] - df["g"]
    df["g_r"] = df["g"] - df["r"]
    df["r_i"] = df["r"] - df["i"]
    df["i_z"] = df["i"] - df["z"]

    return df


train_fe = add_color_features(train_df)
test_fe = add_color_features(test_df)

display(train_fe.head())

In [ ]:
#设置特征列和目标列
TARGET_COL = "class"
ID_COL = "id"

BASE_FEATURES = ["u", "g", "r", "i", "z", "redshift", "alpha", "delta"]
COLOR_FEATURES = ["u_g", "g_r", "r_i", "i_z"]

FEATURE_COLS = BASE_FEATURES + COLOR_FEATURES

missing_train_cols = [col for col in FEATURE_COLS if col not in train_fe.columns]
missing_test_cols = [col for col in FEATURE_COLS if col not in test_fe.columns]

if TARGET_COL not in train_fe.columns:
    raise ValueError(f"Target column '{TARGET_COL}' not found in train.csv.")

if ID_COL not in test_fe.columns:
    raise ValueError(f"ID column '{ID_COL}' not found in test.csv.")

if missing_train_cols:
    raise ValueError(f"Missing feature columns in train data: {missing_train_cols}")

if missing_test_cols:
    raise ValueError(f"Missing feature columns in test data: {missing_test_cols}")

X = train_fe[FEATURE_COLS].copy()
y = train_fe[TARGET_COL].copy()
X_test = test_fe[FEATURE_COLS].copy()

print("Feature columns:")
print(FEATURE_COLS)

print("X shape:", X.shape)
print("X_test shape:", X_test.shape)

print("Target distribution:")
display(y.value_counts())

In [ ]:
#标签编码
label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

print("Class mapping:")
for class_name, encoded_value in zip(label_encoder.classes_, range(len(label_encoder.classes_))):
    print(f"{encoded_value}: {class_name}")

In [ ]:
#划分训练集和验证集
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded,
)

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)

In [ ]:
#训练LightGBM
lgbm_model = lgb.LGBMClassifier(
    objective="multiclass",
    num_class=len(label_encoder.classes_),
    n_estimators=2000,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=30,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
    n_jobs=-1,
    force_col_wise=True,
    verbose=-1,
)

lgbm_model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="multi_logloss",
    callbacks=[
        lgb.early_stopping(stopping_rounds=100),
        lgb.log_evaluation(period=100),
    ],
)

print("Best iteration:", lgbm_model.best_iteration_)

In [ ]:
#验证集预测与指标计算
y_val_pred = lgbm_model.predict(X_val)

val_accuracy = accuracy_score(y_val, y_val_pred)
val_balanced_accuracy = balanced_accuracy_score(y_val, y_val_pred)
val_macro_f1 = f1_score(y_val, y_val_pred, average="macro")

metrics_df = pd.DataFrame(
    {
        "model": ["LightGBM_color"],
        "accuracy": [val_accuracy],
        "balanced_accuracy": [val_balanced_accuracy],
        "macro_f1": [val_macro_f1],
        "best_iteration": [lgbm_model.best_iteration_],
    }
)

display(metrics_df)

metrics_path = TABLE_DIR / "lgbm_metrics.csv"
metrics_df.to_csv(metrics_path, index=False)

print("Metrics saved to:", metrics_path)

In [ ]:
#分类报告
target_names = label_encoder.classes_

report_dict = classification_report(
    y_val,
    y_val_pred,
    target_names=target_names,
    output_dict=True,
)

report_df = pd.DataFrame(report_dict).T
display(report_df)

report_path = TABLE_DIR / "lgbm_classification_report.csv"
report_df.to_csv(report_path, index=True)

print("Classification report saved to:", report_path)

In [ ]:
#混淆矩阵
cm = confusion_matrix(y_val, y_val_pred)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=target_names,
)
disp.plot(ax=ax, values_format="d")
ax.set_title("LightGBM Confusion Matrix")
plt.tight_layout()

confusion_matrix_path = FIGURE_DIR / "lgbm_confusion_matrix.png"
plt.savefig(confusion_matrix_path, dpi=300, bbox_inches="tight")
plt.show()

print("Confusion matrix saved to:", confusion_matrix_path)

In [ ]:
#特征重要性
feature_importance_df = pd.DataFrame(
    {
        "feature": FEATURE_COLS,
        "importance": lgbm_model.feature_importances_,
    }
).sort_values("importance", ascending=False)

display(feature_importance_df)

feature_importance_path = TABLE_DIR / "lgbm_feature_importance.csv"
feature_importance_df.to_csv(feature_importance_path, index=False)

plt.figure(figsize=(8, 5))
plt.barh(
    feature_importance_df["feature"].iloc[::-1],
    feature_importance_df["importance"].iloc[::-1],
)
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("LightGBM Feature Importance")
plt.tight_layout()

feature_importance_fig_path = FIGURE_DIR / "lgbm_feature_importance.png"
plt.savefig(feature_importance_fig_path, dpi=300, bbox_inches="tight")
plt.show()

print("Feature importance table saved to:", feature_importance_path)
print("Feature importance figure saved to:", feature_importance_fig_path)

In [ ]:
#生成LightGBM模型报告
report_md = f"""# LightGBM Model Report

## Model Setting

This notebook trains a LightGBM classifier using the original photometric features and additional color index features.

## Feature Set

Base features:

{BASE_FEATURES}

Color index features:

{COLOR_FEATURES}

Final feature set:

{FEATURE_COLS}

## Validation Results

| Model | Accuracy | Balanced Accuracy | Macro F1 | Best Iteration |
|---|---:|---:|---:|---:|
| LightGBM + color features | {val_accuracy:.6f} | {val_balanced_accuracy:.6f} | {val_macro_f1:.6f} | {lgbm_model.best_iteration_} |

## Output Files

- `tables/lgbm_metrics.csv`
- `tables/lgbm_classification_report.csv`
- `tables/lgbm_feature_importance.csv`
- `figures/lgbm_confusion_matrix.png`
- `figures/lgbm_feature_importance.png`

## Notes

The Kaggle score should be recorded separately after submitting the generated submission file.
"""

report_path = REPORT_DIR / "lgbm_model_report.md"

with open(report_path, "w", encoding="utf-8") as f:
    f.write(report_md)

print("Report saved to:", report_path)

In [ ]:
#训练最终模型
final_lgbm_model = lgb.LGBMClassifier(
    objective="multiclass",
    num_class=len(label_encoder.classes_),
    n_estimators=lgbm_model.best_iteration_ if lgbm_model.best_iteration_ else 1000,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=30,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
    n_jobs=-1,
    force_col_wise=True,
    verbose=-1,
)

final_lgbm_model.fit(X, y_encoded)

print("Final LightGBM model has been trained on the full training dataset.")